In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

2026-03-28 16:08:21.010757: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774714101.201222      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774714101.259814      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774714101.753200      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774714101.753238      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774714101.753240      55 computation_placer.cc:177] computation placer alr

In [4]:
# 1. Load Dataset
# Make sure your dataset path is correct and it has columns: "text" and "label"
df = pd.read_csv("/kaggle/input/datasets/mrsikandarali/metahate-dataset/your_dataset_updated.tsv", sep='\t')  # or use .csv

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (1101165, 2)
   label                                               text
0      0  !!! RT @mayasolovely: As a woman you shouldn't...
1      0  !!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2      0  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3      0  !!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4      0  !!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [5]:
# 2. Clean Text
def clean_text(text):
    text = str(text).lower()                        # Lowercase
    text = re.sub(r"http\S+", "", text)            # Remove URLs
    text = re.sub(r"@\w+", "", text)               # Remove mentions
    text = re.sub(r"#\w+", "", text)               # Remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)        # Remove special characters
    text = re.sub(r"\s+", " ", text).strip()       # Remove extra spaces
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [6]:
# 3. Encode Labels
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])  # 0 = non-hate, 1 = hate

# 4. Tokenization
MAX_WORDS = 50000   # Vocabulary size
MAX_LEN = 100       # Max words per text

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])

sequences = tokenizer.texts_to_sequences(df['clean_text'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')



In [7]:
# 5. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, df['label_enc'].values, test_size=0.2, random_state=42, stratify=df['label_enc']
)

# 6. Create TensorFlow Dataset (efficient for big data)
BATCH_SIZE = 1024
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)



I0000 00:00:1774699244.285181      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774699244.291162      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [13]:

import tensorflow as tf
print(tf.__version__)


2.19.0


In [15]:
from tensorflow.keras.layers import Input

inputs = Input(shape=(MAX_LEN,))
x = Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM)(inputs)
x = Bidirectional(LSTM(64, return_sequences=True))(x)
x = LSTM(32)(x)
x = Dropout(0.5)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 100, 128)       │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 100, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 32)             │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,520,513 (24.87 MB)

 Trainable params: 6,520,513 (24.87 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# 8. Train the Model
EPOCHS = 5  # Increase if resources allow
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS
)

# 9. Evaluate Model
loss, accuracy = model.evaluate(test_dataset)
print("Test Accuracy:", accuracy)

# 10. Save Model
model.save("hate_speech_rnn_model.h5")

Epoch 1/5
861/861 ━━━━━━━━━━━━━━━━━━━━ 68s 76ms/step - accuracy: 0.7886 - loss: 0.5006 - val_accuracy: 0.8603 - val_loss: 0.3137
Epoch 2/5
861/861 ━━━━━━━━━━━━━━━━━━━━ 64s 75ms/step - accuracy: 0.8622 - loss: 0.3105 - val_accuracy: 0.8709 - val_loss: 0.2921
Epoch 3/5
861/861 ━━━━━━━━━━━━━━━━━━━━ 65s 75ms/step - accuracy: 0.8778 - loss: 0.2779 - val_accuracy: 0.8697 - val_loss: 0.2983
Epoch 4/5
861/861 ━━━━━━━━━━━━━━━━━━━━ 65s 75ms/step - accuracy: 0.8906 - loss: 0.2511 - val_accuracy: 0.8687 - val_loss: 0.3191
Epoch 5/5
861/861 ━━━━━━━━━━━━━━━━━━━━ 65s 75ms/step - accuracy: 0.9030 - loss: 0.2251 - val_accuracy: 0.8717 - val_loss: 0.3484
216/216 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.8715 - loss: 0.3506


Test Accuracy: 0.8716813325881958


In [17]:
# 11. Make Predictions on Test Set
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Predict probabilities
y_pred_prob = model.predict(test_dataset)
# Convert probabilities to class labels (0 or 1)
y_pred = (y_pred_prob > 0.5).astype(int)

# Get true labels from test_dataset
y_true = np.concatenate([y for x, y in test_dataset], axis=0)

# 12. Print Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Non-Hate', 'Hate']))

# 13. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

216/216 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step
Classification Report:
              precision    recall  f1-score   support

    Non-Hate       0.91      0.93      0.92    173575
        Hate       0.71      0.66      0.69     46658

    accuracy                           0.87    220233
   macro avg       0.81      0.80      0.80    220233
weighted avg       0.87      0.87      0.87    220233

Confusion Matrix:
[[160976  12599]
 [ 15661  30997]]


# On balalnce dataset 

In [15]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        dfbalance = pd.read_csv("/kaggle/input/datasets/mrsikandarali/balanced-dataset-metahate/balanced_dataset.csv")

texts = dfbalance['text'].astype(str)
labels = dfbalance['label']

# show updated dataset
print(dfbalance.head())
print("Dataset shape:", dfbalance.shape)

/kaggle/input/datasets/mrsikandarali/metahate-dataset/your_dataset_updated.tsv
/kaggle/input/datasets/mrsikandarali/balanced-dataset-metahate/balanced_dataset.csv
/kaggle/input/datasets/mrsikandarali/balanced-dataset/balanced_dataset.csv
   label                                               text
0      1  "@Blackman38Tide: @WhaleLookyHere @HowdyDowdy1...
1      1  "@CB_Baby24: @white_thunduh alsarabsss" hes a ...
2      1  "@DevilGrimz: @VigxRArts you're fucking gay, b...
3      1  "@MarkRoundtreeJr: LMFAOOOO I HATE BLACK PEOPL...
4      1  "@NoChillPaz: "At least I'm not a nigger" http...
Dataset shape: (466578, 2)


In [16]:
# Import necessary libraries
print("Dataset shape:", dfbalance.shape)
print(dfbalance.head())

# 2. Clean Text
def clean_text(text):
    text = str(text).lower()                        # Lowercase
    text = re.sub(r"http\S+", "", text)            # Remove URLs
    text = re.sub(r"@\w+", "", text)               # Remove mentions
    text = re.sub(r"#\w+", "", text)               # Remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)        # Remove special characters
    text = re.sub(r"\s+", " ", text).strip()       # Remove extra spaces
    return text

dfbalance['clean_text'] = dfbalance['text'].apply(clean_text)

# 3. Encode Labels
le = LabelEncoder()
dfbalance['label_enc'] = le.fit_transform(df['label'])  # 0 = non-hate, 1 = hate

# 4. Tokenization
MAX_WORDS = 50000   # Vocabulary size
MAX_LEN = 100       # Max words per text

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])

sequences = tokenizer.texts_to_sequences(df['clean_text'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# 5. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, dfbalance['label_enc'].values, test_size=0.2, random_state=42, stratify=df['label_enc']
)





Dataset shape: (466578, 2)
   label                                               text
0      1  "@Blackman38Tide: @WhaleLookyHere @HowdyDowdy1...
1      1  "@CB_Baby24: @white_thunduh alsarabsss" hes a ...
2      1  "@DevilGrimz: @VigxRArts you're fucking gay, b...
3      1  "@MarkRoundtreeJr: LMFAOOOO I HATE BLACK PEOPL...
4      1  "@NoChillPaz: "At least I'm not a nigger" http...


In [17]:
# 6. Create TensorFlow Dataset (efficient for big data)
BATCH_SIZE = 1024
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)


In [18]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [20]:
from tensorflow.keras.layers import Input

inputs = Input(shape=(MAX_LEN,))
x = Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM)(inputs)
x = Bidirectional(LSTM(64, return_sequences=True))(x)
x = LSTM(32)(x)
x = Dropout(0.5)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 100, 128)       │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 100, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,520,513 (24.87 MB)

 Trainable params: 6,520,513 (24.87 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
# 8. Train the Model
EPOCHS = 5  # Increase if resources allow
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS
)

# 9. Evaluate Model
loss, accuracy = model.evaluate(test_dataset)
print("Test Accuracy:", accuracy)

# 10. Save Model
model.save("hate_speech_rnn_model.h5")

Epoch 1/5
365/365 ━━━━━━━━━━━━━━━━━━━━ 30s 73ms/step - accuracy: 0.5675 - loss: 0.6744 - val_accuracy: 0.6780 - val_loss: 0.6393
Epoch 2/5
365/365 ━━━━━━━━━━━━━━━━━━━━ 28s 76ms/step - accuracy: 0.6840 - loss: 0.6187 - val_accuracy: 0.6651 - val_loss: 0.6404
Epoch 3/5
365/365 ━━━━━━━━━━━━━━━━━━━━ 27s 74ms/step - accuracy: 0.6786 - loss: 0.6189 - val_accuracy: 0.7761 - val_loss: 0.4879
Epoch 4/5
365/365 ━━━━━━━━━━━━━━━━━━━━ 26s 72ms/step - accuracy: 0.7974 - loss: 0.4582 - val_accuracy: 0.8278 - val_loss: 0.3968
Epoch 5/5
365/365 ━━━━━━━━━━━━━━━━━━━━ 27s 74ms/step - accuracy: 0.8432 - loss: 0.3750 - val_accuracy: 0.8330 - val_loss: 0.3889
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.8341 - loss: 0.3870


Test Accuracy: 0.8329547047615051


In [23]:
# 11. Make Predictions on Test Set
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Predict probabilities
y_pred_prob = model.predict(test_dataset)
# Convert probabilities to class labels (0 or 1)
y_pred = (y_pred_prob > 0.5).astype(int)

# Get true labels from test_dataset
y_true = np.concatenate([y for x, y in test_dataset], axis=0)

# 12. Print Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Non-Hate', 'Hate']))

# 13. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step
Classification Report:
              precision    recall  f1-score   support

    Non-Hate       0.86      0.80      0.83     46658
        Hate       0.81      0.87      0.84     46658

    accuracy                           0.83     93316
   macro avg       0.83      0.83      0.83     93316
weighted avg       0.83      0.83      0.83     93316

Confusion Matrix:
[[37346  9312]
 [ 6276 40382]]
